# 02 Baseline Logistic Regression

This notebook mirrors the reusable baseline pipeline in `src/` and gives you a place to inspect preprocessing choices and baseline metrics.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from src.fraud_detection.data import load_train_data, split_features_target
from src.fraud_detection.features import build_preprocessor, drop_high_missing_columns
from src.fraud_detection.metrics import compute_classification_metrics

In [ ]:
data = load_train_data(sample_size=50000)
X, y = split_features_target(data)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, dropped_columns = drop_high_missing_columns(X_train, threshold=0.95)
X_valid = X_valid.drop(columns=dropped_columns, errors="ignore")

preprocessor, numeric_features, categorical_features = build_preprocessor(X_train)
len(numeric_features), len(categorical_features), len(dropped_columns)

In [ ]:
baseline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                solver="liblinear",
                random_state=42,
            ),
        ),
    ]
)

baseline.fit(X_train, y_train)
valid_scores = baseline.predict_proba(X_valid)[:, 1]
compute_classification_metrics(y_valid, valid_scores)